## 1. 데이터 불러오기
> - 데이터는 Pandas 라이브러리의 pd.read_csv 함수를 활용하여 불러온다.

In [ ]:
#필요 라이브러리 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#한글 폰트 지정
import matplotlib
import matplotlib.font_manager as fm

fm.get_fontconfig_fonts()
font_location = 'C:/Windows/Fonts/malgun.ttf' # For Windows
font_name = fm.FontProperties(fname=font_location).get_name()
matplotlib.rc('font', family=font_name)

In [ ]:
#데이터 경로 지정
order_data_file_path = "./data/order_detail.csv"
customer_data_file_path = "./data/customer_data.csv"

#데이터 불러오기
order_data = pd.read_csv(order_data_file_path)
customer_data = pd.read_csv(customer_data_file_path)

In [ ]:
order_data

In [ ]:
#주문일자 컬럼을 생성한다.
order_data["주문일자"] = order_data["주문일시"].str.slice(start=0, stop=10)
order_data["주문일자"] = pd.to_datetime(order_data["주문일자"], format="%Y-%m-%d")

## 2. RFM 고객데이터 만들기

In [ ]:
order_data.head()

In [ ]:
import datetime as dt
오늘날짜 = dt.datetime(2024,9,1) #데이터 추출 기준 8월 마감 기준

In [ ]:
rfm = order_data.groupby("고객번호").agg({'주문일자' : lambda 주문일자 : (오늘날짜 - 주문일자.max()).days,
                                  '주문번호' : lambda 주문번호 : 주문번호.nunique(),
                                  '총주문금액_판매가' : lambda 총주문금액_판매가 : 총주문금액_판매가.sum()}) 

In [ ]:
rfm.columns = ['recency','frequency','monetary']
rfm = rfm.reset_index()
rfm.head()

In [ ]:
customer_data = pd.merge(customer_data, rfm, on="고객번호", how="left")

In [ ]:
customer_data.head()

In [ ]:
customer_data = customer_data.dropna()
customer_data = customer_data.reset_index(drop=True)
customer_data.head()

In [ ]:
customer_data.plot(x="나이", y="frequency", c="monetary", kind="scatter")

## 3. 군집분석 수행하기 (Kmeans)

In [ ]:
from sklearn.preprocessing import scale
from sklearn.cluster import KMeans

In [ ]:
target_df = customer_data[["나이","frequency","monetary","recency"]]
target_df.head(10)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# min-max 정규화 적용
scaler.fit(target_df)
target_df_scaled = scaler.transform(target_df)

In [ ]:
target_df_scaled = pd.DataFrame(target_df_scaled, columns=["나이","frequency","monetary","recency"])

In [ ]:
kmeans = KMeans(n_clusters=3, init='k-means++', max_iter=300,random_state= 992)
kmeans.fit(target_df_scaled)

In [ ]:
target_df_scaled["cluster"] = kmeans.labels_

In [ ]:
target_df_scaled["cluster"].value_counts()

In [ ]:
customer_data_vec= target_df_scaled[["나이","recency","frequency","monetary"]]

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
pca_transformed = pca.fit_transform(target_df_scaled)

target_df_scaled['pca_x'] = pca_transformed[:,0]
target_df_scaled['pca_y'] = pca_transformed[:,1]

#도출된 군집을 기준으로 인덱스 값 추출
ind_0 = target_df_scaled[target_df_scaled["cluster"]==0].index
ind_1 = target_df_scaled[target_df_scaled["cluster"]==1].index
ind_2 = target_df_scaled[target_df_scaled["cluster"]==2].index

plt.scatter(x=target_df_scaled.loc[ind_0,'pca_x'], y=target_df_scaled.loc[ind_0,'pca_y'], marker='o') 
plt.scatter(x=target_df_scaled.loc[ind_1,'pca_x'], y=target_df_scaled.loc[ind_1,'pca_y'], marker='D')
plt.scatter(x=target_df_scaled.loc[ind_2,'pca_x'], y=target_df_scaled.loc[ind_2,'pca_y'], marker='v')

plt.show()


In [ ]:
target_df.loc[ind_0,"cluster"] = 0
target_df.loc[ind_1,"cluster"] = 1
target_df.loc[ind_2,"cluster"] = 2

In [ ]:
target_df.groupby(["cluster"]).agg({"나이":"mean","frequency":"mean","monetary":"mean","recency":"mean"})

In [ ]:
# <실루엣 계수>
# 실루엣 계수가 0에 가까울 수록 근처 군집과 가깝다는 의미이며,
# 1에 가까울 수록 근처 군집과 멀다는 의미이다.
# 음수인 경우 잘못된 군집에 할당되었다는 의미이다.
from sklearn.metrics import silhouette_score

average_score  = silhouette_score(target_df_scaled, target_df_scaled["cluster"])

In [ ]:
average_score

## 4. 군집분석 수행하기 (DBSCAN)

In [ ]:
target_df = customer_data[["나이","frequency","monetary","recency"]]
target_df.head(10)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# StandardScaler 정규화 적용
scaler.fit(target_df)
target_df_scaled = scaler.transform(target_df)

In [ ]:
target_df_scaled = pd.DataFrame(target_df_scaled, columns=["나이","frequency","monetary","recency"])

In [ ]:
from sklearn.cluster import DBSCAN

# epsilon, min_sample 개수 설정
dbscan = DBSCAN(eps=0.3, min_samples=100)

# 군집화 모델 학습 및 클러스터 예측 결과 반환
dbscan.fit(target_df_scaled)
target_df_scaled['cluster'] = dbscan.fit_predict(target_df_scaled)

In [ ]:
target_df

In [ ]:
target_df_scaled["cluster"].value_counts()

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
pca_transformed = pca.fit_transform(target_df_scaled)

target_df_scaled['pca_x'] = pca_transformed[:,0]
target_df_scaled['pca_y'] = pca_transformed[:,1]

#도출된 군집을 기준으로 인덱스 값 추출
ind_0 = target_df_scaled[target_df_scaled["cluster"]==0].index
ind_1 = target_df_scaled[target_df_scaled["cluster"]==-1].index
ind_2 = target_df_scaled[target_df_scaled["cluster"]==1].index
ind_3 = target_df_scaled[target_df_scaled["cluster"]==2].index


plt.scatter(x=target_df_scaled.loc[ind_0,'pca_x'], y=target_df_scaled.loc[ind_0,'pca_y'], marker='o') 
plt.scatter(x=target_df_scaled.loc[ind_1,'pca_x'], y=target_df_scaled.loc[ind_1,'pca_y'], marker='D')
plt.scatter(x=target_df_scaled.loc[ind_2,'pca_x'], y=target_df_scaled.loc[ind_2,'pca_y'], marker='v')
plt.scatter(x=target_df_scaled.loc[ind_3,'pca_x'], y=target_df_scaled.loc[ind_3,'pca_y'], marker='^')

plt.show()


In [ ]:
target_df.loc[ind_0,"cluster"] = 0
target_df.loc[ind_1,"cluster"] = -1
target_df.loc[ind_2,"cluster"] = 1
target_df.loc[ind_3,"cluster"] = 2

In [ ]:
#지수표현식 없애기 
pd.options.display.float_format = '{:.0f}'.format
target_df.groupby(["cluster"]).agg({"나이":"mean","frequency":"mean","monetary":"mean","recency":"mean"})

In [ ]:
# <실루엣 계수>
# 실루엣 계수가 0에 가까울 수록 근처 군집과 가깝다는 의미이며,
# 1에 가까울 수록 근처 군집과 멀다는 의미이다.
# 음수인 경우 잘못된 군집에 할당되었다는 의미이다.
from sklearn.metrics import silhouette_score

average_score  = silhouette_score(target_df_scaled, target_df_scaled["cluster"])

In [ ]:
average_score